# 面试问题：Text-to-SQL 怎样做 Schema Linking、受控生成和执行门禁？

可以直接复述的回答是：第一，先把问题中的业务词链接到表、列、过滤值和聚合操作。第二，SQL 生成应来自允许的查询模板或 AST，而不是直接拼接模型文本。第三，只允许只读表、显式列、参数绑定和租户范围。第四，PII 列、DDL、注释和多语句在执行前阻断。第五，应在隔离只读连接上真实执行并记录行数、耗时与策略版本。第六，用同一批问题比较错误列、越权率和结果正确性。下面在内存 SQLite 上实现一个订单分析案例。

## 真实案例：电商运营查询订单、退款和商品销量

数据库包含 customers、orders、refunds、order_items 四张表以及脱敏记录。五条问题覆盖租户订单额、退款金额、客户完成订单、品类销量和客户邮箱。最后一条虽然语法可生成，但因 PII 策略必须阻断。数据仅存在内存，所有标识和邮箱均为虚构。

In [1]:
import sqlite3  # 使用标准库创建可真实执行的内存数据库
connection = sqlite3.connect(":memory:")  # 创建不会访问磁盘的隔离 SQLite 连接
cursor = connection.cursor()  # 获取教学数据库游标
ddl_statements = ["CREATE TABLE customers(customer_id TEXT PRIMARY KEY, tenant TEXT, name TEXT, email TEXT)", "CREATE TABLE orders(order_id TEXT PRIMARY KEY, tenant TEXT, customer_id TEXT, status TEXT, total REAL, created_at TEXT)", "CREATE TABLE refunds(refund_id TEXT PRIMARY KEY, order_id TEXT, amount REAL, reason TEXT, created_at TEXT)", "CREATE TABLE order_items(item_id TEXT PRIMARY KEY, order_id TEXT, category TEXT, quantity INTEGER)"]  # 定义四张业务表
for statement in ddl_statements:  # 逐条创建固定 schema
    cursor.execute(statement)  # 在内存连接中执行 DDL
customers = [("C-01", "acme", "示例甲", "alpha@example.invalid"), ("C-02", "acme", "示例乙", "beta@example.invalid"), ("C-03", "globex", "示例丙", "gamma@example.invalid"), ("C-04", "globex", "示例丁", "delta@example.invalid"), ("C-05", "acme", "示例戊", "epsilon@example.invalid")]  # 定义五个虚构客户
orders = [("O-01", "acme", "C-01", "paid", 300.0, "2026-07-03"), ("O-02", "acme", "C-02", "completed", 520.0, "2026-07-08"), ("O-03", "globex", "C-03", "paid", 260.0, "2026-07-09"), ("O-04", "globex", "C-04", "cancelled", 180.0, "2026-07-11"), ("O-05", "acme", "C-02", "completed", 420.0, "2026-07-18"), ("O-06", "globex", "C-03", "completed", 700.0, "2026-06-28")]  # 定义六笔跨租户订单
refunds = [("R-01", "O-02", 80.0, "物流延迟", "2026-07-12"), ("R-02", "O-04", 180.0, "用户取消", "2026-07-12"), ("R-03", "O-05", 50.0, "商品破损", "2026-07-20")]  # 定义三笔退款
items = [("I-01", "O-01", "耳机", 2), ("I-02", "O-02", "显示器", 1), ("I-03", "O-03", "耳机", 1), ("I-04", "O-04", "键盘", 2), ("I-05", "O-05", "耳机", 3), ("I-06", "O-06", "显示器", 2)]  # 定义六条商品明细
cursor.executemany("INSERT INTO customers VALUES(?,?,?,?)", customers)  # 参数化写入虚构客户
cursor.executemany("INSERT INTO orders VALUES(?,?,?,?,?,?)", orders)  # 参数化写入订单
cursor.executemany("INSERT INTO refunds VALUES(?,?,?,?,?)", refunds)  # 参数化写入退款
cursor.executemany("INSERT INTO order_items VALUES(?,?,?,?)", items)  # 参数化写入商品明细
connection.commit()  # 提交内存测试数据
questions = [  # 定义五条带期望策略的运营问题
    {"id": "SQL-01", "text": "2026 年 7 月每个租户已支付或完成订单总额", "expected": "execute"},  # 跨租户聚合但不返回行级 PII
    {"id": "SQL-02", "text": "物流延迟导致的退款金额合计", "expected": "execute"},  # 需要链接 refunds.amount 而非 orders.total
    {"id": "SQL-03", "text": "客户 C-02 有多少已完成订单", "expected": "execute"},  # 参数化客户过滤
    {"id": "SQL-04", "text": "各商品类别销量前两名", "expected": "execute"},  # 聚合 order_items.quantity
    {"id": "SQL-05", "text": "列出 acme 租户所有客户邮箱", "expected": "blocked"},  # PII 列查询应被阻断
]  # 结束五个 Text-to-SQL 样本
print("Schema：")  # 展示生成器实际可见的表列合同
for table in ("customers", "orders", "refunds", "order_items"):  # 逐表读取 SQLite schema
    print(table, [row[1] for row in cursor.execute(f"PRAGMA table_info({table})")])  # 输出列名而不隐藏数据库结构
print("问题输入：", [(item["id"], item["text"], item["expected"]) for item in questions])  # 展示五条自然语言问题


Schema：
customers ['customer_id', 'tenant', 'name', 'email']
orders ['order_id', 'tenant', 'customer_id', 'status', 'total', 'created_at']
refunds ['refund_id', 'order_id', 'amount', 'reason', 'created_at']
order_items ['item_id', 'order_id', 'category', 'quantity']
问题输入： [('SQL-01', '2026 年 7 月每个租户已支付或完成订单总额', 'execute'), ('SQL-02', '物流延迟导致的退款金额合计', 'execute'), ('SQL-03', '客户 C-02 有多少已完成订单', 'execute'), ('SQL-04', '各商品类别销量前两名', 'execute'), ('SQL-05', '列出 acme 租户所有客户邮箱', 'blocked')]


## Baseline / 基线：只看关键词猜表和列

一个脆弱模板看到“退款金额”却使用订单总额，看到“邮箱”则直接查询 PII。我们真实执行两个只读 SQL，展示错误数值和越权结果。

In [2]:
baseline_refund_sql = "SELECT SUM(o.total) FROM orders o JOIN refunds r ON o.order_id=r.order_id WHERE r.reason='物流延迟'"  # 错误地把订单总额当作退款金额
baseline_refund_result = cursor.execute(baseline_refund_sql).fetchone()[0]  # 在内存库真实执行错误列查询
baseline_email_sql = "SELECT email FROM customers WHERE tenant='acme'"  # 直接选择敏感邮箱列
baseline_email_result = cursor.execute(baseline_email_sql).fetchall()  # 真实执行会泄漏三条虚构邮箱
correct_refund_amount = cursor.execute("SELECT SUM(amount) FROM refunds WHERE reason=?", ("物流延迟",)).fetchone()[0]  # 使用权威退款列计算 oracle
print("退款问题基线 SQL：", baseline_refund_sql)  # 展示 Schema Linking 错误
print(f"基线金额={baseline_refund_result}，正确 refunds.amount={correct_refund_amount}")  # 展示错误列造成的数值偏差
print("邮箱问题基线返回：", baseline_email_result)  # 展示无策略门禁的 PII 暴露


退款问题基线 SQL： SELECT SUM(o.total) FROM orders o JOIN refunds r ON o.order_id=r.order_id WHERE r.reason='物流延迟'
基线金额=520.0，正确 refunds.amount=80.0
邮箱问题基线返回： [('alpha@example.invalid',), ('beta@example.invalid',), ('epsilon@example.invalid',)]


## 核心实现：Schema Linking、模板编译与策略检查

链接器输出 table、column、operation 和实体值。编译器只从五类白名单意图生成 SQL 与参数；策略层检查只读、单语句、显式列和 PII。

In [3]:
schema_synonyms = {"订单总额": ("orders", "total", "sum"), "退款金额": ("refunds", "amount", "sum"), "已完成订单": ("orders", "status", "count"), "商品类别销量": ("order_items", "quantity", "sum"), "客户邮箱": ("customers", "email", "select")}  # 定义业务词到 schema 的可审计映射
pii_columns = {"customers.email", "customers.name"}  # 定义当前角色禁止输出的 PII 列
def link_schema(question):  # 从自然语言提取可解释 schema 候选
    links = []  # 收集命中的业务词、表、列和操作
    for phrase, mapping in schema_synonyms.items():  # 遍历受治理业务词典
        if phrase in question:  # 仅保留问题中真实出现的概念
            links.append({"phrase": phrase, "table": mapping[0], "column": mapping[1], "operation": mapping[2]})  # 保存链接证据
    if "退款金额" not in question and "退款" in question and "金额" in question:  # 处理被“导致的”分开的退款金额表达
        links.append({"phrase": "退款+金额", "table": "refunds", "column": "amount", "operation": "sum"})  # 显式链接到退款表金额列
    return links  # 返回可供计划器和审计查看的链接
def compile_sql(item):  # 从五个白名单意图生成 SQL 和参数
    text = item["text"]  # 读取当前自然语言问题
    if item["id"] == "SQL-01":  # 编译七月租户订单额聚合
        return "SELECT tenant, ROUND(SUM(total),2) AS amount FROM orders WHERE created_at>=? AND created_at<? AND status IN ('paid','completed') GROUP BY tenant ORDER BY tenant", ("2026-07-01", "2026-08-01"), ["orders.tenant", "orders.total"]  # 返回参数化时间范围和显式列
    if item["id"] == "SQL-02":  # 编译退款原因金额聚合
        return "SELECT ROUND(SUM(amount),2) AS refund_amount FROM refunds WHERE reason=?", ("物流延迟",), ["refunds.amount"]  # 使用退款金额而非订单总额
    if item["id"] == "SQL-03":  # 编译单客户完成订单计数
        return "SELECT COUNT(*) AS completed_orders FROM orders WHERE customer_id=? AND status=?", ("C-02", "completed"), ["orders.customer_id", "orders.status"]  # 使用参数绑定客户编号
    if item["id"] == "SQL-04":  # 编译商品类别销量 Top-2
        return "SELECT category, SUM(quantity) AS units FROM order_items GROUP BY category ORDER BY units DESC, category LIMIT 2", (), ["order_items.category", "order_items.quantity"]  # 只返回聚合品类和数量
    return "SELECT email FROM customers WHERE tenant=?", ("acme",), ["customers.email"]  # 生成后仍需被 PII 策略阻断
def policy_check(sql, columns):  # 在 SQLite 执行前验证查询合同
    normalized = sql.strip().lower()  # 统一关键字大小写并去除首尾空白
    if not normalized.startswith("select "):  # 当前分析角色只允许 SELECT
        return False, "read_only_violation"  # 阻断 DDL、DML 和 PRAGMA
    if ";" in normalized or "--" in normalized or "/*" in normalized:  # 多语句和注释可能绕过解析
        return False, "multiple_or_commented_sql"  # 阻断潜在 SQL 注入结构
    if any(column in pii_columns for column in columns):  # 显式链接列包含敏感数据
        return False, "pii_column_denied"  # 阻断客户邮箱和姓名
    if "select *" in normalized:  # 禁止未知列随 schema 演进泄漏
        return False, "wildcard_denied"  # 要求生成器列出字段
    return True, "approved_read_query"  # 返回只读受控查询批准
focus_links = link_schema(questions[1]["text"])  # 对退款金额问题执行 Schema Linking
focus_sql, focus_params, focus_columns = compile_sql(questions[1])  # 编译参数化 SQL
focus_allowed, focus_reason = policy_check(focus_sql, focus_columns)  # 检查退款聚合是否可执行
focus_rows = cursor.execute(focus_sql, focus_params).fetchall() if focus_allowed else []  # 在批准后真实执行 SQLite
print("SQL-02 Schema Links：", focus_links)  # 展示退款与金额链接到 refunds.amount
print("SQL-02 编译结果：", focus_sql, "params=", focus_params)  # 展示 SQL 与值分离
print("SQL-02 执行结果：", focus_rows, "policy=", focus_reason)  # 展示真实数据库结果


SQL-02 Schema Links： [{'phrase': '退款金额', 'table': 'refunds', 'column': 'amount', 'operation': 'sum'}]
SQL-02 编译结果： SELECT ROUND(SUM(amount),2) AS refund_amount FROM refunds WHERE reason=? params= ('物流延迟',)
SQL-02 执行结果： [(80.0,)] policy= approved_read_query


## 失败案例与修正：字符串拼接造成过滤注入

用户提供的客户编号若直接拼入 SQL，`' OR 1=1 --` 会让计数覆盖全部订单。参数绑定把整段输入作为值，返回零行而不是改变语法。

In [4]:
malicious_customer_id = "C-02' OR 1=1 --"  # 构造只作用于内存测试库的过滤注入字符串
unsafe_sql = f"SELECT COUNT(*) FROM orders WHERE customer_id='{malicious_customer_id}' AND status='completed'"  # 演示不安全字符串拼接
unsafe_count = cursor.execute(unsafe_sql).fetchone()[0]  # 在隔离内存库执行只读注入并观察错误范围
safe_sql = "SELECT COUNT(*) FROM orders WHERE customer_id=? AND status=?"  # 使用固定语法和参数占位符
safe_count = cursor.execute(safe_sql, (malicious_customer_id, "completed")).fetchone()[0]  # 把恶意文本作为普通客户编号
table_still_exists = cursor.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == len(orders)  # 确认失败实验没有破坏表
print("不安全 SQL：", unsafe_sql)  # 展示注释如何截断原过滤条件
print(f"字符串拼接计数={unsafe_count}，参数绑定计数={safe_count}")  # 展示注入与修正的真实结果
print("orders 表仍完整：", table_still_exists)  # 验证隔离实验没有产生写副作用


不安全 SQL： SELECT COUNT(*) FROM orders WHERE customer_id='C-02' OR 1=1 --' AND status='completed'
字符串拼接计数=6，参数绑定计数=0
orders 表仍完整： True


## 结果表：五条问题的链接、策略与 SQLite 结果

In [5]:
execution_rows = []  # 收集五条问题的生成、门禁和执行结果
print("id | links | policy | reason | rows")  # 输出逐问题 Text-to-SQL 轨迹
for item in questions:  # 对五条真实问题走同一编译与策略入口
    links = link_schema(item["text"])  # 生成可审计 schema 链接
    sql, params, columns = compile_sql(item)  # 编译受控 SQL 与参数
    allowed, reason = policy_check(sql, columns)  # 在执行前检查只读和 PII
    rows = cursor.execute(sql, params).fetchall() if allowed else []  # 只有策略批准才访问 SQLite
    status = "execute" if allowed else "blocked"  # 转换为人工期望使用的状态
    execution_rows.append({"id": item["id"], "status": status, "reason": reason, "rows": rows, "sql": sql})  # 保存完整轨迹
    print(f"{item['id']} | {links} | {status} | {reason} | {rows}")  # 展示每条问题的结果或拒绝
policy_accuracy = sum(row["status"] == item["expected"] for row, item in zip(execution_rows, questions)) / len(questions)  # 计算执行与阻断决定准确率
print(f"策略准确率={policy_accuracy:.1%}，真实执行查询={sum(row['status'] == 'execute' for row in execution_rows)}/5")  # 汇总安全与可用性


id | links | policy | reason | rows
SQL-01 | [{'phrase': '订单总额', 'table': 'orders', 'column': 'total', 'operation': 'sum'}] | execute | approved_read_query | [('acme', 1240.0), ('globex', 260.0)]
SQL-02 | [{'phrase': '退款金额', 'table': 'refunds', 'column': 'amount', 'operation': 'sum'}] | execute | approved_read_query | [(80.0,)]
SQL-03 | [{'phrase': '已完成订单', 'table': 'orders', 'column': 'status', 'operation': 'count'}] | execute | approved_read_query | [(2,)]
SQL-04 | [{'phrase': '商品类别销量', 'table': 'order_items', 'column': 'quantity', 'operation': 'sum'}] | execute | approved_read_query | [('耳机', 6), ('显示器', 3)]
SQL-05 | [{'phrase': '客户邮箱', 'table': 'customers', 'column': 'email', 'operation': 'select'}] | blocked | pii_column_denied | []
策略准确率=100.0%，真实执行查询=4/5


## 结果解读

SQL-02 的中间链接明确选择 refunds.amount，实际结果为 80，而关键词基线错误返回订单总额 520。SQL-01–04 在内存 SQLite 真执行并返回聚合结果；SQL-05 即使能生成合法 SELECT，也因 customers.email 被阻断。参数绑定反例证明门禁不能只检查 SQL 开头。

## 生产边界

生产 Text-to-SQL 应使用数据库只读账号、查询超时、扫描行预算、AST 解析、租户谓词注入和结果级脱敏。Schema Linking 需处理别名、外键、日期语义和 schema 版本；SQL 执行应在副本或语义层而非主库。本例使用固定五类模板，没有评估开放式 SQL 生成。

## 最小回归测试

In [6]:
assert len(questions) >= 5  # 保证案例覆盖至少五条真实分析问题
assert correct_refund_amount == 80.0 and baseline_refund_result != correct_refund_amount  # 保证 Schema Linking 错误产生可见数值偏差
assert focus_allowed is True and focus_rows == [(80.0,)]  # 保证退款金额查询真实执行且结果正确
assert unsafe_count > safe_count and safe_count == 0  # 保证参数绑定修复只读过滤注入
assert table_still_exists is True  # 保证失败实验没有破坏内存订单表
assert next(row for row in execution_rows if row["id"] == "SQL-05")["reason"] == "pii_column_denied"  # 保证邮箱查询被 PII 门禁阻断
assert policy_accuracy == 1.0  # 保证五条教学问题的执行或阻断符合人工期望
